# P8 — Hard Negatives corregidos

Compara un control **1:1 con negativos aleatorios** contra un escenario **1:1 con hard negatives emparejados por similitud histórica**. Para cada positivo futuro se selecciona, sin reemplazo, un verdadero negativo con la menor distancia en el espacio estandarizado de las siete características históricas seleccionadas en P5. El futuro solo se usa para definir casos y controles; no entra en las características ni en la distancia de matching.


In [1]:
from pathlib import Path
from itertools import combinations
from collections import Counter
import random
import numpy as np
import pandas as pd
import networkx as nx
import features as ft

# =============================================================================
# P8. Hard negatives para evaluacion temporal de link prediction
# =============================================================================
# Objetivo:
#   Comparar, manteniendo TODO lo demas fijo, dos esquemas 1:1:
#     (a) negativos aleatorios
#     (b) hard negatives emparejados: verdaderos negativos futuros cuya
#         representacion HISTORICA es lo mas parecida posible a la de los positivos.
#
# Esta version corrige el primer diseno de hard negatives, que maximizaba
# proximidad estructural y genero una clase negativa demasiado homogenea.
#
# IMPORTANTE:
#   - threshold de coautoria fijo en 1.
#   - las etiquetas se determinan con el horizonte futuro.
#   - la similitud para el matching se calcula exclusivamente con features historicas.
#   - el futuro solo determina que pares son positivos/negativos del horizonte;
#     NO entra en las features ni en la distancia de matching.
# =============================================================================

BASE_DIR = Path('C:/Users/Dra. Pilar/UltimaversionProgramasComparacion')
OUT_DIR = BASE_DIR / 'P8_hard_negatives'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 41
MIN_COAUTHORED_PAPERS = 1
VAL_SIZE = 0.20
NEGATIVE_RATIO = 1

random.seed(SEED)
np.random.seed(SEED)

FILES = {
    'full_2000_2020': BASE_DIR / 'BD_UNAM_Graph_2000_2020.gml',
    'sub_2000_2020':  BASE_DIR / 'BD_UNAM_Sub_Graph_2000_2020.gml',
    'full_2021_2022': BASE_DIR / 'BD_UNAM_Graph_2021_2022.gml',
    'sub_2021_2022':  BASE_DIR / 'BD_UNAM_Sub_Graph_2021_2022.gml',
    'full_2023_2024': BASE_DIR / 'BD_UNAM_Graph_2023_2024.gml',
    'sub_2023_2024':  BASE_DIR / 'BD_UNAM_Sub_Graph_2023_2024.gml',
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan archivos GML:\n' + '\n'.join(missing))

G_2000_2020 = nx.read_gml(FILES['full_2000_2020'])
S_2000_2020 = nx.read_gml(FILES['sub_2000_2020'])
G_2021_2022 = nx.read_gml(FILES['full_2021_2022'])
S_2021_2022 = nx.read_gml(FILES['sub_2021_2022'])
G_2023_2024 = nx.read_gml(FILES['full_2023_2024'])
S_2023_2024 = nx.read_gml(FILES['sub_2023_2024'])

G_HIST_2022 = nx.compose(G_2000_2020, G_2021_2022)
S_HIST_2022 = nx.compose(S_2000_2020, S_2021_2022)

print('Grafos cargados:')
for name, G in [
    ('G_2000_2020', G_2000_2020),
    ('G_2021_2022', G_2021_2022),
    ('G_2023_2024', G_2023_2024),
    ('G_HIST_2022', G_HIST_2022),
]:
    print(f'{name}: nodos={G.number_of_nodes():,}, aristas={G.number_of_edges():,}')


def node_type(G, node):
    return str(G.nodes[node].get('tipo', '')).strip().lower()


def is_article_node(G, node):
    return node_type(G, node) == 'articulo'


def is_author_node(G, node):
    return node_type(G, node) == 'autor'


def author_nodes(G):
    return {n for n in G.nodes if is_author_node(G, n)}


def add_isolated_historical_authors(S_history, G_history):
    S = S_history.copy()
    hist_authors = author_nodes(G_history)
    missing_authors = hist_authors - set(S.nodes())
    S.add_nodes_from(missing_authors)
    return S, missing_authors


def canon_pair(u, v):
    return tuple(sorted((str(u), str(v))))


def coauthorship_counts(G, eligible_authors=None):
    eligible = None if eligible_authors is None else set(eligible_authors)
    counts = Counter()
    for article in G.nodes:
        if not is_article_node(G, article):
            continue
        authors = [
            a for a in G.neighbors(article)
            if is_author_node(G, a) and (eligible is None or a in eligible)
        ]
        authors = sorted(set(authors), key=str)
        for u, v in combinations(authors, 2):
            counts[canon_pair(u, v)] += 1
    return counts


def threshold_edges(G, threshold, eligible_authors=None):
    counts = coauthorship_counts(G, eligible_authors)
    return {pair for pair, n in counts.items() if n >= threshold}


def safe_feature(func, graph, u, v, default=np.nan):
    try:
        if u not in graph or v not in graph:
            return default
        value = func(graph, u, v)
        return default if value is None else value
    except Exception:
        return default


def safe_len_feature(func, graph, u, v, default=0):
    try:
        if u not in graph or v not in graph:
            return default
        value = func(graph, u, v)
        return default if value is None else len(value)
    except Exception:
        return default


def paper_count(G, author):
    if author not in G:
        return 0
    return sum(1 for n in G.neighbors(author) if is_article_node(G, n))


def shortest_path_safe(G, u, v):
    try:
        if u not in G or v not in G:
            return np.nan
        return nx.shortest_path_length(G, u, v)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return np.nan


def pair_structural_hardness(S_history, u, v):
    """Componentes de dificultad calculados SOLO con historia."""
    if u not in S_history or v not in S_history:
        return {
            'hard_cn': 0.0,
            'hard_jaccard': 0.0,
            'hard_ra': 0.0,
            'hard_aa': 0.0,
            'hard_shortest_path': np.nan,
            'hard_inv_path': 0.0,
        }

    try:
        cn = float(sum(1 for _ in nx.common_neighbors(S_history, u, v)))
    except Exception:
        cn = 0.0

    try:
        jac = float(next(nx.jaccard_coefficient(S_history, [(u, v)]))[2])
    except Exception:
        jac = 0.0

    try:
        ra = float(next(nx.resource_allocation_index(S_history, [(u, v)]))[2])
    except Exception:
        ra = 0.0

    try:
        aa = float(next(nx.adamic_adar_index(S_history, [(u, v)]))[2])
    except Exception:
        aa = 0.0

    sp = shortest_path_safe(S_history, u, v)
    inv_path = 0.0 if pd.isna(sp) or sp <= 0 else 1.0 / float(sp)

    return {
        'hard_cn': cn,
        'hard_jaccard': jac,
        'hard_ra': ra,
        'hard_aa': aa,
        'hard_shortest_path': sp,
        'hard_inv_path': inv_path,
    }


def add_hardness_score(pool):
    """Agrega score por ranking. No usa etiquetas futuras mas alla de saber que el pool son negativos."""
    out = pool.copy()
    score_cols = ['hard_cn', 'hard_jaccard', 'hard_ra', 'hard_aa', 'hard_inv_path']
    rank_cols = []
    for c in score_cols:
        rc = f'{c}_rankpct'
        # percentile rank: mayor valor historico = mayor dificultad
        out[rc] = out[c].rank(method='average', pct=True)
        rank_cols.append(rc)
    out['hardness_score'] = out[rank_cols].mean(axis=1)
    return out


def build_candidate_pools(G_history, G_future, S_history, threshold=1):
    """
    Construye positivos nuevos y TODOS los verdaderos negativos del horizonte.
    El futuro solo determina la etiqueta; la dificultad de los negativos se calcula
    exclusivamente con S_history.
    """
    hist_authors = author_nodes(G_history)
    future_authors = author_nodes(G_future)
    eligible = sorted(hist_authors.intersection(future_authors), key=str)

    if len(eligible) < 2:
        raise ValueError('No hay suficientes autores elegibles.')

    if any(not is_author_node(G_history, n) for n in eligible):
        raise AssertionError('La cohorte contiene nodos no autor en historia.')
    if any(not is_author_node(G_future, n) for n in eligible):
        raise AssertionError('La cohorte contiene nodos no autor en futuro.')

    hist_positive = threshold_edges(G_history, threshold, eligible_authors=eligible)
    future_positive_all = threshold_edges(G_future, threshold, eligible_authors=eligible)
    new_positives = sorted(future_positive_all - hist_positive)
    positive_set = set(new_positives)

    if not new_positives:
        raise ValueError('No se encontraron enlaces positivos nuevos.')

    # Negativos verdaderos del horizonte: no existian en historia y tampoco aparecen
    # como positivos durante el horizonte objetivo.
    negative_pairs = []
    for u, v in combinations(eligible, 2):
        pair = canon_pair(u, v)
        if pair in hist_positive or pair in positive_set:
            continue
        negative_pairs.append(pair)

    if len(negative_pairs) < len(new_positives):
        raise ValueError('No hay suficientes negativos para construir muestra 1:1.')

    # Calcular dificultad SOLO con historia.
    hard_rows = []
    for u, v in negative_pairs:
        d = pair_structural_hardness(S_history, u, v)
        hard_rows.append({'source': u, 'target': v, **d})
    neg_pool = add_hardness_score(pd.DataFrame(hard_rows))

    audit = {
        'historical_authors': len(hist_authors),
        'future_authors': len(future_authors),
        'eligible_authors': len(eligible),
        'historical_positive_pairs': len(hist_positive),
        'future_positive_pairs_all': len(future_positive_all),
        'new_future_positive_pairs': len(new_positives),
        'true_negative_pool': len(negative_pairs),
        'threshold': threshold,
    }
    return new_positives, neg_pool, audit


def select_random_negatives(neg_pool, n, seed):
    if n > len(neg_pool):
        raise ValueError(f'Se solicitaron {n} negativos y solo existen {len(neg_pool)}.')
    return neg_pool.sample(n=n, random_state=seed).copy().reset_index(drop=True)


def make_balanced_sample(positives, selected_negatives, seed):
    pos_df = pd.DataFrame(positives, columns=['source', 'target'])
    pos_df['connected'] = 1
    neg_df = selected_negatives[['source', 'target']].copy()
    neg_df['connected'] = 0
    out = pd.concat([pos_df, neg_df], ignore_index=True)
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)


def split_development_matched(sample, seed):
    """
    Mantiene el mismo reparto de positivos entre random y hard.
    Para 17+17 y 20% produce 14/13 en train y 3/4 en val, como P4.
    """
    pos = sample[sample['connected'] == 1].sort_values(['source', 'target']).reset_index(drop=True)
    neg = sample[sample['connected'] == 0].sort_values(['source', 'target']).reset_index(drop=True)

    rng_pos = np.random.RandomState(seed)
    rng_neg = np.random.RandomState(seed + 1)
    pos_idx = rng_pos.permutation(len(pos))
    neg_idx = rng_neg.permutation(len(neg))

    n_val_total = int(np.ceil(VAL_SIZE * len(sample)))
    n_val_pos = max(1, int(np.floor(VAL_SIZE * len(pos))))
    n_val_neg = n_val_total - n_val_pos
    n_val_neg = max(1, min(n_val_neg, len(neg) - 1))

    val_pos = pos.iloc[pos_idx[:n_val_pos]]
    train_pos = pos.iloc[pos_idx[n_val_pos:]]
    val_neg = neg.iloc[neg_idx[:n_val_neg]]
    train_neg = neg.iloc[neg_idx[n_val_neg:]]

    train = pd.concat([train_pos, train_neg], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    val = pd.concat([val_pos, val_neg], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return train, val


def add_historical_features(df, G_history, S_history):
    out = df.copy()

    missing_structural = (set(out['source']) | set(out['target'])) - set(S_history.nodes())
    if missing_structural:
        raise AssertionError(f'Faltan {len(missing_structural)} autores historicos en S_history.')

    out['sum_of_papers'] = out.apply(
        lambda r: paper_count(G_history, r['source']) + paper_count(G_history, r['target']), axis=1
    )
    out['sum_of_keywords'] = out.apply(
        lambda r: safe_len_feature(ft.sum_keywords, G_history, r['source'], r['target'], 0), axis=1
    )
    out['keywords_match'] = out.apply(
        lambda r: safe_len_feature(ft.keyword_match, G_history, r['source'], r['target'], 0), axis=1
    )
    out['sum_of_areas'] = out.apply(
        lambda r: safe_len_feature(ft.sum_areas, G_history, r['source'], r['target'], 0), axis=1
    )

    out['sum_of_neighbors'] = out.apply(
        lambda r: len(ft.neighbors(G_history, r['source'])) + len(ft.neighbors(G_history, r['target']))
        if r['source'] in G_history and r['target'] in G_history else 0,
        axis=1,
    )
    out['log_secundary_neighbors'] = out.apply(
        lambda r: safe_feature(ft.sum_sec_neig, G_history, r['source'], r['target'], 0.0), axis=1
    )
    out['lenght_short_path'] = out.apply(
        lambda r: shortest_path_safe(S_history, r['source'], r['target']), axis=1
    )
    out['clustering_index_sum'] = out.apply(
        lambda r: safe_feature(ft.clustering_index, S_history, r['source'], r['target'], np.nan), axis=1
    )
    out['jaccard_coefficient'] = out.apply(
        lambda r: safe_feature(ft.jaccard_coefficient, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['resource_allocation'] = out.apply(
        lambda r: safe_feature(ft.get_resource_allocation_index, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['adamic_adar_index'] = out.apply(
        lambda r: safe_feature(ft.get_adamic_adar_index, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['preferential_attachment'] = out.apply(
        lambda r: safe_feature(ft.get_preferential_attachment, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['community_cn'] = out.apply(
        lambda r: safe_feature(ft.community_cn, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['community_ra'] = out.apply(
        lambda r: safe_feature(ft.community_ra, S_history, r['source'], r['target'], 0.0), axis=1
    )
    out['no_historical_path'] = out['lenght_short_path'].isna().astype(int)
    return out


MATCH_FEATURES = [
    'sum_of_neighbors',
    'sum_of_papers',
    'sum_of_keywords',
    'community_cn',
    'keywords_match',
    'adamic_adar_index',
    'resource_allocation',
]


def select_matched_hard_negatives(positives, neg_pool, G_history, S_history, seed):
    """
    Selecciona un negativo por cada positivo mediante matching 1:1 sin reemplazo.

    La distancia se calcula en el espacio estandarizado de las mismas siete
    caracteristicas HISTORICAS seleccionadas en P5. El scaler y la imputacion se
    estiman con el pool de candidatos negativos del mismo corte historico.

    El conocimiento del horizonte futuro solo se usa para definir casos (positivos)
    y controles verdaderamente negativos; no entra en ninguna feature.
    """
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import pairwise_distances
    from scipy.optimize import linear_sum_assignment

    pos_df = pd.DataFrame(positives, columns=['source', 'target'])
    pos_df['connected'] = 1
    pos_feat = add_historical_features(pos_df, G_history, S_history)

    neg_base = neg_pool[['source', 'target']].copy()
    neg_base['connected'] = 0
    neg_feat = add_historical_features(neg_base, G_history, S_history)

    if len(neg_feat) < len(pos_feat):
        raise ValueError('No hay suficientes negativos para matching 1:1.')

    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    X_neg_imp = imputer.fit_transform(neg_feat[MATCH_FEATURES])
    X_neg = scaler.fit_transform(X_neg_imp)
    X_pos = scaler.transform(imputer.transform(pos_feat[MATCH_FEATURES]))

    # Pequeno jitter reproducible solo para desempatar distancias exactamente iguales.
    rng = np.random.RandomState(seed)
    D = pairwise_distances(X_pos, X_neg, metric='euclidean')
    D = D + rng.uniform(0.0, 1e-12, size=D.shape)

    row_ind, col_ind = linear_sum_assignment(D)
    selected = neg_feat.iloc[col_ind].copy().reset_index(drop=True)
    selected['matched_positive_index'] = row_ind
    selected['match_distance'] = D[row_ind, col_ind]

    # Reincorporar los diagnosticos estructurales del pool original.
    diagnostics = neg_pool[['source', 'target', 'hard_cn', 'hard_jaccard', 'hard_ra',
                            'hard_aa', 'hard_shortest_path', 'hard_inv_path',
                            'hardness_score']].copy()
    selected = selected.merge(diagnostics, on=['source', 'target'], how='left')
    return selected


def matching_stats(selected_negatives, prefix):
    if 'match_distance' not in selected_negatives.columns:
        return {
            f'{prefix}_match_distance_mean': np.nan,
            f'{prefix}_match_distance_median': np.nan,
            f'{prefix}_match_distance_max': np.nan,
        }
    return {
        f'{prefix}_match_distance_mean': float(selected_negatives['match_distance'].mean()),
        f'{prefix}_match_distance_median': float(selected_negatives['match_distance'].median()),
        f'{prefix}_match_distance_max': float(selected_negatives['match_distance'].max()),
    }


def hardness_stats(selected_negatives, prefix):
    sp = selected_negatives['hard_shortest_path']
    finite_sp = sp.dropna()
    return {
        f'{prefix}_n': len(selected_negatives),
        f'{prefix}_hardness_mean': float(selected_negatives['hardness_score'].mean()),
        f'{prefix}_hardness_median': float(selected_negatives['hardness_score'].median()),
        f'{prefix}_cn_mean': float(selected_negatives['hard_cn'].mean()),
        f'{prefix}_fraction_cn_gt0': float((selected_negatives['hard_cn'] > 0).mean()),
        f'{prefix}_fraction_path_le3': float((sp <= 3).fillna(False).mean()),
        f'{prefix}_shortest_path_mean_finite': float(finite_sp.mean()) if len(finite_sp) else np.nan,
        f'{prefix}_fraction_no_path': float(sp.isna().mean()),
    }


# ----------------------------------------------------------------------------
# Preparacion de subgrafos historicos con autores aislados
# ----------------------------------------------------------------------------
S_2000_2020, ISOLATED_2000_2020 = add_isolated_historical_authors(S_2000_2020, G_2000_2020)
S_HIST_2022, ISOLATED_2000_2022 = add_isolated_historical_authors(S_HIST_2022, G_HIST_2022)

print(f'Autores historicos anadidos como aislados 2000-2020: {len(ISOLATED_2000_2020):,}')
print(f'Autores historicos anadidos como aislados 2000-2022: {len(ISOLATED_2000_2022):,}')

FEATURE_COLUMNS = [
    'sum_of_papers', 'sum_of_neighbors', 'log_secundary_neighbors',
    'sum_of_keywords', 'keywords_match', 'sum_of_areas',
    'lenght_short_path', 'clustering_index_sum', 'jaccard_coefficient',
    'resource_allocation', 'adamic_adar_index', 'preferential_attachment',
    'community_cn', 'community_ra', 'no_historical_path'
]

# ----------------------------------------------------------------------------
# Pool de desarrollo: historia 2000-2020 -> futuro 2021-2022
# ----------------------------------------------------------------------------
print('\nConstruyendo pool de desarrollo...')
dev_positives, dev_neg_pool, audit_dev = build_candidate_pools(
    G_2000_2020, G_2021_2022, S_2000_2020, threshold=MIN_COAUTHORED_PAPERS
)

# Pool de test: historia 2000-2022 -> futuro 2023-2024
print('Construyendo pool de test...')
test_positives, test_neg_pool, audit_test = build_candidate_pools(
    G_HIST_2022, G_2023_2024, S_HIST_2022, threshold=MIN_COAUTHORED_PAPERS
)

n_dev_neg = len(dev_positives) * NEGATIVE_RATIO
n_test_neg = len(test_positives) * NEGATIVE_RATIO

summary_rows = []
audit_rows = []
selected_cache = {}

for mode in ['random', 'hard']:
    print('\n' + '=' * 72)
    print(f'NEGATIVE MODE = {mode.upper()} | RATIO 1:1')
    print('=' * 72)

    if mode == 'random':
        dev_neg = select_random_negatives(dev_neg_pool, n_dev_neg, seed=SEED)
        test_neg = select_random_negatives(test_neg_pool, n_test_neg, seed=SEED + 1)
    else:
        dev_neg = select_matched_hard_negatives(
            dev_positives, dev_neg_pool, G_2000_2020, S_2000_2020, seed=SEED
        )
        test_neg = select_matched_hard_negatives(
            test_positives, test_neg_pool, G_HIST_2022, S_HIST_2022, seed=SEED + 1
        )
    selected_cache[(mode, 'dev')] = dev_neg
    selected_cache[(mode, 'test')] = test_neg

    development = make_balanced_sample(dev_positives, dev_neg, seed=SEED)
    sample_train, sample_val = split_development_matched(development, seed=SEED)
    sample_test = make_balanced_sample(test_positives, test_neg, seed=SEED + 1)

    # Features: exclusivamente con historia
    sample_train = add_historical_features(sample_train, G_2000_2020, S_2000_2020)
    sample_val = add_historical_features(sample_val, G_2000_2020, S_2000_2020)
    sample_test = add_historical_features(sample_test, G_HIST_2022, S_HIST_2022)

    required = {'source', 'target', 'connected', *FEATURE_COLUMNS}
    for split_name, df in [('train', sample_train), ('val', sample_val), ('test', sample_test)]:
        missing_cols = required - set(df.columns)
        if missing_cols:
            raise AssertionError(f'{mode} {split_name}: faltan columnas {missing_cols}')
        if df.duplicated(['source', 'target']).any():
            raise AssertionError(f'{mode} {split_name}: hay pares duplicados.')

    # Archivos
    suffix = f'thr1_neg1_{mode}_isolates'
    train_file = OUT_DIR / f'sample_temporal_train_hist2000_2020_target2021_2022_{suffix}.csv'
    val_file = OUT_DIR / f'sample_temporal_val_hist2000_2020_target2021_2022_{suffix}.csv'
    test_file = OUT_DIR / f'sample_temporal_test_hist2000_2022_target2023_2024_{suffix}.csv'

    sample_train.to_csv(train_file, index=False)
    sample_val.to_csv(val_file, index=False)
    sample_test.to_csv(test_file, index=False)

    def cls(df):
        vc = df['connected'].value_counts()
        return int(vc.get(1, 0)), int(vc.get(0, 0))

    tr_p, tr_n = cls(sample_train)
    va_p, va_n = cls(sample_val)
    te_p, te_n = cls(sample_test)

    stats_dev = hardness_stats(dev_neg, 'dev_neg')
    stats_test = hardness_stats(test_neg, 'test_neg')

    summary_rows.append({
        'negative_mode': mode,
        'ratio': '1:1',
        'threshold': MIN_COAUTHORED_PAPERS,
        'development_positive': len(dev_positives),
        'development_negative': len(dev_neg),
        'train_n': len(sample_train),
        'train_positive': tr_p,
        'train_negative': tr_n,
        'val_n': len(sample_val),
        'val_positive': va_p,
        'val_negative': va_n,
        'test_positive': te_p,
        'test_negative': te_n,
        'test_n': len(sample_test),
        **stats_dev,
        **stats_test,
        **matching_stats(dev_neg, 'dev_neg'),
        **matching_stats(test_neg, 'test_neg'),
        'train_file': train_file.name,
        'val_file': val_file.name,
        'test_file': test_file.name,
    })

    for split_name, hist, horizon, aud, negs in [
        ('train_val', '2000-2020', '2021-2022', audit_dev, dev_neg),
        ('test', '2000-2022', '2023-2024', audit_test, test_neg),
    ]:
        audit_rows.append({
            'experiment': 'P8_hard_negatives',
            'negative_mode': mode,
            'ratio': '1:1',
            'split': split_name,
            'feature_history': hist,
            'label_horizon': horizon,
            'threshold': MIN_COAUTHORED_PAPERS,
            'seed': SEED if split_name == 'train_val' else SEED + 1,
            'future_used_for_features': False,
            'future_used_for_hardness_score': False,
            'hardness_definition': 'hard = 1:1 minimum-distance matching to future positives in standardized historical Top-7 feature space; random = random true negatives',
            'matching_features': '|'.join(MATCH_FEATURES),
            **aud,
            **hardness_stats(negs, 'selected_negative'),
            **matching_stats(negs, 'selected_negative'),
        })

    print(f'Train: n={len(sample_train)}, positivos={tr_p}, negativos={tr_n}')
    print(f'Val:   n={len(sample_val)}, positivos={va_p}, negativos={va_n}')
    print(f'Test:  n={len(sample_test)}, positivos={te_p}, negativos={te_n}')
    print(f'Hardness media dev={stats_dev["dev_neg_hardness_mean"]:.4f} | test={stats_test["test_neg_hardness_mean"]:.4f}')
    if mode == 'hard':
        print(f'Distancia matching media dev={dev_neg["match_distance"].mean():.4f} | test={test_neg["match_distance"].mean():.4f}')

# Overlap entre conjuntos negativos random y hard
for split_key, label in [('dev', 'development'), ('test', 'test')]:
    rnd = set(map(tuple, selected_cache[('random', split_key)][['source', 'target']].to_numpy()))
    hrd = set(map(tuple, selected_cache[('hard', split_key)][['source', 'target']].to_numpy()))
    overlap = len(rnd & hrd)
    for row in summary_rows:
        row[f'{label}_random_hard_overlap_n'] = overlap
        row[f'{label}_random_hard_overlap_fraction'] = overlap / max(1, len(hrd))

summary_df = pd.DataFrame(summary_rows)
audit_df = pd.DataFrame(audit_rows)

summary_file = OUT_DIR / 'P8_hard_negative_summary.csv'
audit_file = OUT_DIR / 'P8_hard_negative_audit.csv'
dev_pool_file = OUT_DIR / 'P8_development_negative_pool_with_hardness.csv'
test_pool_file = OUT_DIR / 'P8_test_negative_pool_with_hardness.csv'

summary_df.to_csv(summary_file, index=False)
audit_df.to_csv(audit_file, index=False)
dev_neg_pool.to_csv(dev_pool_file, index=False)
test_neg_pool.to_csv(test_pool_file, index=False)

print('\n' + '=' * 72)
print('RESUMEN HARD NEGATIVES')
print('=' * 72)
show_cols = [
    'negative_mode', 'development_positive', 'development_negative',
    'dev_neg_hardness_mean', 'dev_neg_fraction_cn_gt0', 'dev_neg_fraction_path_le3', 'dev_neg_match_distance_mean',
    'test_positive', 'test_negative',
    'test_neg_hardness_mean', 'test_neg_fraction_cn_gt0', 'test_neg_fraction_path_le3', 'test_neg_match_distance_mean'
]
print(summary_df[show_cols].to_string(index=False))

print('\nArchivos resumen:')
print(summary_file)
print(audit_file)
print('\nPools de negativos con score de dificultad:')
print(dev_pool_file)
print(test_pool_file)
print('\nCSV individuales guardados en:')
print(OUT_DIR)


C:\Users\pilarang\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Grafos cargados:
G_2000_2020: nodos=9,041, aristas=14,064
G_2021_2022: nodos=5,151, aristas=6,752
G_2023_2024: nodos=3,207, aristas=4,025
G_HIST_2022: nodos=13,371, aristas=20,728
Autores historicos anadidos como aislados 2000-2020: 0
Autores historicos anadidos como aislados 2000-2022: 0

Construyendo pool de desarrollo...
Construyendo pool de test...

NEGATIVE MODE = RANDOM | RATIO 1:1
Train: n=27, positivos=14, negativos=13
Val:   n=7, positivos=3, negativos=4
Test:  n=112, positivos=56, negativos=56
Hardness media dev=0.4668 | test=0.5059

NEGATIVE MODE = HARD | RATIO 1:1
Train: n=27, positivos=14, negativos=13
Val:   n=7, positivos=3, negativos=4
Test:  n=112, positivos=56, negativos=56
Hardness media dev=0.5945 | test=0.6015
Distancia matching media dev=0.8040 | test=0.5132

RESUMEN HARD NEGATIVES
negative_mode  development_positive  development_negative  dev_neg_hardness_mean  dev_neg_fraction_cn_gt0  dev_neg_fraction_path_le3  dev_neg_match_distance_mean  test_positive  test_ne